# VS code cũng đẹp mà
Tui yêu cái đẹp, thấy cái gì đẹp là tui lưu lại hết à
Tui muốn sản phẩm mk làm ra có sự chỉnh chu, áp dụng được vào thực tiễn, thực sự có ích cho đời

In [2]:
import random

# https://www.fileformat.info/info/unicode/category/So/list.htm

list = ['''Dừa lòng :)) \U0001F965''',
        '\U0001F335 \U0001F335 \U0001F335',
        'Tóc giả giả \U0001FAAD',
        '9094 -> Mồm\U0001F415 vó\U0001F434',
        'Kay \U0001F332 \U0001F333 \U0001F99D \U0001F423',
        'Tăng Phúc \U0001F9A6',
        '2 Neko \U0001F451 \U0001F40D \U0001F63A \U0001F638 \U0001F639 \U0001F640',
        'Nui \U0001F35C',
        'ST \U0001F43A \U0001F427', 
        '\U0001F33B', '\U0001F337', '\U0001F338', '\U0001F339', '\U0001F33A', '\U0001F33C', '\U0001F33E', '\U0001F340']
# print(*list, sep='\n')

random_item = (random.choice(list) + ' \u2740 '+ random.choice(list) + ' \u2740 '+random.choice(list) + ' \u2740 ' + '\n')*5
print(random_item)

Kay 🌲 🌳 🦝 🐣 ❀ 🌼 ❀ 🍀 ❀ 
Kay 🌲 🌳 🦝 🐣 ❀ 🌼 ❀ 🍀 ❀ 
Kay 🌲 🌳 🦝 🐣 ❀ 🌼 ❀ 🍀 ❀ 
Kay 🌲 🌳 🦝 🐣 ❀ 🌼 ❀ 🍀 ❀ 
Kay 🌲 🌳 🦝 🐣 ❀ 🌼 ❀ 🍀 ❀ 



In [1]:
# import libraries 

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

import time
import schedule

import requests
import re
import datetime
import smtplib

import csv 
import numpy as np
import pandas as pd

# cần 2 cái dưới để điều hướng con trỏ chuột sang trang kế
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException

import logging # dùng để ghi tiến trình & lỗi khi crawl data

In [2]:
# --- Thiết lập logging ---
logging.basicConfig(
    filename="crawl_jobs.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding='utf-8'
)

# 1. Lấy link job trong 1 page

In [3]:

# Khởi tạo trình duyệt Chrome
driver = webdriver.Chrome()

# Mở trang vietnamworks (trang 1)
url = 'https://www.vietnamworks.com/viec-lam?q=data&g=5&page=1'
driver.get(url)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn


# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [4]:
# Lấy mã nguồn HTML của trang web
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [ ]:
soup

In [ ]:
# Tìm container chứa danh sách công việc
job_data = soup.find("div", class_="st")

In [8]:
# Lấy danh sách job
job_previews = job_data.find_all('a', class_="img_job_card")
print(f"Found {len(job_previews)} job listings on page 1")

Found 50 job listings on page 1


In [ ]:
job_previews

In [14]:
job_previews[0]

<a class="img_job_card" href="/senior-data-scientist-2027481-jv?source=searchResults&amp;searchType=2&amp;placement=2027481&amp;sortBy=date&amp;qs=0" rel="noreferrer" target="_blank" title="Senior Data Scientist"><div class="sc-cHHTbD exPzkH"><span style="box-sizing: border-box; display: block; overflow: hidden; width: initial; height: initial; background: none; opacity: 1; border: 0px; margin: 0px; padding: 0px; position: absolute; inset: 0px;"><img alt="Senior Data Scientist" data-nimg="fill" decoding="async" sizes="100vw" src="https://cdn.vietnamworks.com/images?url=https%3A%2F%2Fimages.vietnamworks.com%2Fpictureofcompany%2Fd4%2F11437144.jpg&amp;w=3840&amp;q=75" srcset="https://cdn.vietnamworks.com/images?url=https%3A%2F%2Fimages.vietnamworks.com%2Fpictureofcompany%2Fd4%2F11437144.jpg&amp;w=640&amp;q=75 640w, https://cdn.vietnamworks.com/images?url=https%3A%2F%2Fimages.vietnamworks.com%2Fpictureofcompany%2Fd4%2F11437144.jpg&amp;w=750&amp;q=75 750w, https://cdn.vietnamworks.com/image

In [17]:
# Lấy danh sách link URL dẫn job
links = job_previews[0].get('href')
links

'/senior-data-scientist-2027481-jv?source=searchResults&searchType=2&placement=2027481&sortBy=date&qs=0'

# 2. Con trỏ chuột sang trang kế tiếp

In [19]:
# Tìm container chứa các nút button chuyển trang
soup.find("ul", class_="pagination pagination-square justify-content-center")

<ul class="pagination pagination-square justify-content-center"><li class="page-item active"><button class="sc-gRtvSG dWlGKP btn-info btn-lg" type="button">1</button></li><li class="page-item btn-default"><button class="sc-gRtvSG hYzJLY btn-default btn-md clickable" type="button">2</button></li><li class="page-item btn-default"><button class="sc-gRtvSG hYzJLY btn-default btn-md clickable" type="button">3</button></li><li class="page-item btn-default" data-key="1"><button class="sc-gRtvSG hYzJLY btn-default btn-md clickable" type="button">&gt;</button></li></ul>

In [20]:
# &gt; chỉ tồn tại trong HTML source, còn Selenium làm việc với DOM đã render, nên chỉ thấy > thôi.

buttons = job_data.find_all("button")
for b in buttons:
    print(b.text)   # sẽ in ra >, <, << chứ không in &gt;


1
2
3
>


In [ ]:
all_jobs_data = []
page = 1

while True:
    print(f"=== Đang cào trang {page} ===")
    
    # Lấy HTML của trang hiện tại
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    
    # TODO: Cào link 
    # Tìm container chứa danh sách công việc
    job_data = soup.find("div", class_="block-job-list")    
    
    
    if not job_data:
        print("Could not find job list container")
    
    else:
        # Lấy danh sách các công việc 
        job_items = job_data.find_all('a', class_="img_job_card")
        print(f"Found {len(job_items)} job listings on page {page}")

        for job in job_items:
            try:
                job_link = "https://www.vietnamworks.com" + job.get('href') if job.get('href') else None

                all_jobs_data.append(
                    {
                        "Job Link": job_link,
                    }
                )
            except Exception as e:
                print(f"Error processing job: {e}")


    
    
    try:
        # Tìm nút ">" (Next)
        job_container = driver.find_element(By.CSS_SELECTOR, "div.block-job-list")
        next_button = job_container.find_element(By.XPATH, './/button[text()=">"]')


        # Cuộn xuống dưới trang để hiển thị nút đó rồi mới click
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", next_button)

        page += 1
        
        # Giới hạn số trang (phòng vòng lặp vô tận)
        if page > 10:
            break
        
        
        time.sleep(5)  # chờ load trang mới
        
        # Cuộn xuống để load toàn bộ job
        last_height = driver.execute_script("return document.body.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(3)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
    except NoSuchElementException:
        print("Không còn trang kế tiếp. Đã tới trang cuối cùng!")
        break

driver.quit()

=== Đang cào trang 1 ===
Found 50 job listings on page 1
=== Đang cào trang 2 ===
Found 50 job listings on page 2
Dừng lại đang test 2 trang thôi.


In [22]:
all_jobs_data

[{'Job Link': 'https://www.vietnamworks.com/senior-data-scientist-2027481-jv?source=searchResults&searchType=2&placement=2027481&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.com/phat-trien-ai-topk-kinh-nghiem-3-nam-2021268-jv?source=searchResults&searchType=2&placement=2021268&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.com/ai-data-analytics-2019903-jv?source=searchResults&searchType=2&placement=2019903&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.com/senior-application-and-data-security-officer-2017789-jv?source=searchResults&searchType=2&placement=2017789&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.com/tuyen-dung-nhan-vien-van-hanh-amazon-binh-tan-hcm-2025165-jv?source=searchResults&searchType=2&placement=2025165&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.com/hr-data-analyst-female--2022769-jv?source=searchResults&searchType=2&placement=2022769&sortBy=date&qs=0'},
 {'Job Link': 'https://www.vietnamworks.co

In [23]:
# Lưu dữ liệu công việc vào tệp CSV
csv_file = "vietnamworks_job_listings.csv"
csv_columns = [
    "Job Link",
]

try:
    with open(csv_file, mode="w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=csv_columns)
        writer.writeheader()
        for job in all_jobs_data:
            writer.writerow(job)
    print(f"Data has been saved to {csv_file}")
except IOError:
    print("Error while saving to CSV file")

Data has been saved to vietnamworks_job_listings.csv


In [24]:
len(all_jobs_data)

100

# 3. Bắt đầu lấy data chi tiết nè

In [25]:
# job_link = 'https://www.vietnamworks.com/senior-data-scientist-2027481-jv?source=searchResults&searchType=2&placement=2027481&sortBy=date&qs=0'

job_link = 'https://www.vietnamworks.com/senior-data-scientist-2027481-jv?source=searchResults&searchType=2&placement=2027481&sortBy=date&qs=0'

In [35]:

# Khởi tạo trình duyệt Chrome
driver = webdriver.Chrome()

# Mở trang job chi tiết
driver.get(job_link)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn


# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

## 3.1 Giai đoạn 1 lấy list links trong từng page

In [36]:
driver.get('https://www.vietnamworks.com/viec-lam?q=data&g=5&page=1')
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn


# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [37]:
# Lấy mã nguồn HTML của trang web
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [38]:
# Tìm container chứa danh sách công việc
job_data = soup.find("div", class_="block-job-list")

In [39]:
# Lấy danh sách list_job_links
list_job_links = job_data.find_all('a', attrs={'class': 'img_job_card'})
print(f"Found {len(list_job_links)} job listings on page 1")

Found 50 job listings on page 1


In [40]:
# function 1: Tạo list link trong 1 page

def get_link(soup):
    try:
        # Tìm container chứa danh sách công việc
        job_data = soup.find("div", class_="block-job-list")
        
        # Lấy danh sách list_job_links
        list_job_links_tag = job_data.find_all('a', attrs={'class': 'img_job_card'})

        # Gắn tên miền --> lưu link vào 1 list trống
        all_link = ["https://www.vietnamworks.com" + link.get('href') for link in list_job_links_tag]
        
    except Exception as e:
        print(f"⚠️ Lỗi khi xử lý job: {e}")

    return all_link

In [41]:
# test function: Tạo list link trong 1 page
all_link = get_link(soup)
print(f"Found {len(all_link)} job listings on page 1")
all_link

Found 50 job listings on page 1


['https://www.vietnamworks.com/business-analyst-middlesenior-level-khoi-ai-thiet-bidu-lieusmartcitynen-tang-2028729-jv?source=searchResults&searchType=2&placement=2028729&sortBy=date&qs=0',
 'https://www.vietnamworks.com/senior-data-scientist-2027481-jv?source=searchResults&searchType=2&placement=2027481&sortBy=date&qs=0',
 'https://www.vietnamworks.com/phat-trien-ai-topk-kinh-nghiem-3-nam-2021268-jv?source=searchResults&searchType=2&placement=2021268&sortBy=date&qs=0',
 'https://www.vietnamworks.com/ai-data-analytics-2019903-jv?source=searchResults&searchType=2&placement=2019903&sortBy=date&qs=0',
 'https://www.vietnamworks.com/senior-application-and-data-security-officer-2017789-jv?source=searchResults&searchType=2&placement=2017789&sortBy=date&qs=0',
 'https://www.vietnamworks.com/tuyen-dung-nhan-vien-van-hanh-amazon-binh-tan-hcm-2025165-jv?source=searchResults&searchType=2&placement=2025165&sortBy=date&qs=0',
 'https://www.vietnamworks.com/hr-data-analyst-female--2022769-jv?source=

## 3.2 Cào chi tiết trong vòng lặp ```for link in all_link```

In [35]:
job_link = 'https://www.vietnamworks.com/business-analyst-middlesenior-level-khoi-ai-thiet-bidu-lieusmartcitynen-tang-2028729-jv'

In [42]:
job_link = all_link[0]

In [9]:
job_link

'https://www.vietnamworks.com/business-analyst-middlesenior-level-khoi-ai-thiet-bidu-lieusmartcitynen-tang-2028729-jv?source=searchResults&searchType=2&placement=2028729&sortBy=date&qs=0'

In [36]:
# Khởi tạo trình duyệt Chrome
driver = webdriver.Chrome()

driver.get(job_link)

# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [5]:
try:
    # 1. Tìm nút trực tiếp bằng nội dung chữ của nó
    # (Dùng contains để linh hoạt nếu text có khoảng trắng thừa)
    xem_full_btn = driver.find_element(By.XPATH, "//button[contains(text(), 'Xem đầy đủ mô tả công việc')]")
    
    # 2. Click thẳng bằng JavaScript (không cần cuộn thêm vì bạn đã cuộn trang rồi)
    driver.execute_script("arguments[0].click();", xem_full_btn)
    logging.info("Clicked 'Xem đầy đủ mô tả công việc'!")
    
    # 3. QUAN TRỌNG: Chờ khoảng 1-2 giây để nội dung bung ra hoàn toàn trước khi lấy HTML
    time.sleep(1.5) 

except NoSuchElementException:
    # Nếu không tìm thấy, có thể là do tin này ngắn nên không có nút "Xem đầy đủ"
    logging.warning("Không thấy nút 'Xem đầy đủ', có thể nội dung đã hiển thị hết.")
except Exception as e:
    logging.error(f"Lỗi khi click: {e}")

In [6]:
# điều hướng dùng drive click nút xem thêm ở mục "Thông tin việc làm" 
# Phải mở hết các nút xem thêm trước khi lấy html new_soup gì đó
    
try:
    # tìm nút "Xem thêm" ở mục "Thông tin việc làm"
    xem_them_btn = driver.find_element(
        By.XPATH,
        "//h2[normalize-space()='Thông tin việc làm']/following::button[normalize-space()='Xem thêm'][1]"
    )
    
    # Cuộn xuống dưới trang để hiển thị nút đó rồi mới click
    driver.execute_script("arguments[0].click();", xem_them_btn)
    logging.info("Clicked button 'Xem thêm' in 'Thông tin việc làm'!")

except NoSuchElementException:
    logging.warning("Haven't found button 'Xem thêm' in 'Thông tin việc làm'!")

except Exception as e:
    logging.error(f"Lỗi không xác định khi click 'Xem thêm': {e}")


In [7]:
# điều hướng dùng drive click nút xem thêm ở mục "Phúc lợi" 
    
try:
    # tìm nút "Xem thêm" ở mục "Phúc lợi"
    xem_them_btn = driver.find_element(
        By.XPATH,
        "//h2[normalize-space()='Các phúc lợi dành cho bạn']/following::button[normalize-space()='Xem thêm'][1]"
    )
    
    # Cuộn xuống dưới trang để hiển thị nút đó rồi mới click
    driver.execute_script("arguments[0].click();", xem_them_btn)
    logging.info("Clicked button 'Xem thêm' in 'Các phúc lợi dành cho bạn'!")

except NoSuchElementException:
    logging.warning("Haven't found button 'Xem thêm' in 'Các phúc lợi dành cho bạn'!")

except Exception as e:
    logging.error(f"Lỗi không xác định khi click 'Xem thêm': {e}")

In [37]:
# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

1	Tiêu đề công việc //
2	Công ty  //
3	Địa điểm  //
4	Mức lương
5	Ngày cập nhật mới nhất
6	Thời gian còn lại
7	Ngày hết hạn nộp
8	Ngành nghề
9	Lĩnh vực
10	Hình thức (on-site / remote)
11	Loại công việc ( full / partime )
12	Cấp bậc (nhân viên / intern)
13	năm kinh nghiệm
14	Trình độ học vấn
15	Số lượng tuyển dụng
16	Khung giờ làm việc
17	Kỹ năng
18	Mô tả công việc 
19	Yêu cầu
20	Phúc lợi
21	Từ Khóa (Tag)

### 3.2.1. PHẦN ĐẦU TRANG

#### 3.2.1.1    Urgent

In [38]:
new_soup.find_all('span', attrs={'name': 'label'})

[<span class="sc-ab270149-0 cVbwLK" color="#FF7D55" font-family="Inter" name="label">Thương lượng</span>,
 <span class="sc-ab270149-0 cVbwLK" color="#FF7D55" font-family="Inter" name="label">Thương lượng</span>]

In [33]:
def get_is_urgent(soup):
    """Kiểm tra có nhãn Tuyển gấp / Urgent không từ thẻ label cụ thể."""
    try:
        # Tìm tất cả các thẻ span có name='label' theo format bạn đã khai phá
        labels = soup.find_all('span', attrs={'name': 'label'})
        
        # Duyệt qua từng label một
        for label in labels:
            text = label.get_text(strip=True).lower()  # Ép về chữ thường để so sánh dễ hơn
            if 'urgent' in text or 'tuyển gấp' in text:
                return "Yes"  # Chỉ cần có 1 thẻ thỏa mãn là trả về Yes liền
                
        return "No"  # Quét hết danh sách thẻ mà không có => No
    except Exception:
        return "No"

# --- Chạy thử nghiệm ở Jupyter ---
# Lưu ý: 'new_soup' là biến chứa HTML của trang chi tiết bạn đang test
# print(get_is_urgent(new_soup))


In [39]:
job_urgent = get_is_urgent(new_soup)
job_urgent

'No'

#### 3.2.1.2	Title Job

In [54]:
title = new_soup.find('h1', attrs={'name': 'title'}).text.strip()
title

'Business Analyst (Middle/Senior Level) khối AI Thiết Bị/Dữ liệu/Smartcity/Nền tảng'

In [47]:
# function 2: lấy tiêu đề job

def get_title(new_soup):
    try:
        title = new_soup.find('h1', attrs={'name': 'title'}).text.strip()
        
    except AttributeError:
        title = None

    return title

In [48]:
# test function: lấy tiêu đề job
title = get_title(new_soup)
title

'Business Analyst (Middle/Senior Level) khối AI Thiết Bị/Dữ liệu/Smartcity/Nền tảng'

#### 3.2.1.3	Company

In [49]:
# Lấy tên cty
company_title = new_soup.find('a', attrs={'name': 'label'}).text.strip()
company_title

'CÔNG TY CỔ PHẦN VINSMART FUTURE'

In [50]:
# function 3: lấy tên cty

def get_company(new_soup):
    try:
        company = new_soup.find('a', attrs={'name': 'label'}).text.strip()
        
    except AttributeError:
        company = None

    return company

In [51]:
# test function: lấy tên cty
company = get_company(new_soup)
company

'CÔNG TY CỔ PHẦN VINSMART FUTURE'

#### 3.2.1.4     Local

In [13]:
new_soup.find_all('span', attrs={'name': 'paragraph'})

[<span class="sc-ab270149-0 ePOHWr" color="#000" font-family="Inter" name="paragraph">Hết hạn trong 18 ngày</span>,
 <span class="sc-ab270149-0 ePOHWr" color="#000" font-family="Inter" name="paragraph">622 lượt xem</span>,
 <span class="sc-ab270149-0 ePOHWr" color="#000" font-family="Inter" name="paragraph">Hồ Chí Minh, Hà Nội</span>,
 <span class="sc-ab270149-0 eGWBVW" color="#000" font-family="Inter" name="paragraph" style="text-transform: capitalize;">SAP Senior Consultant - SAP Integrated Business Planning (Ibp) (AMS Project)</span>,
 <span class="sc-ab270149-0 ePOHWr" color="#000" font-family="Inter" name="paragraph">100-499 nhân viên</span>,
 <span class="sc-ab270149-0 eGWBVW" color="#000" font-family="Inter" name="paragraph" style="text-transform: capitalize;">SAP Senior Consultant - SAP Integrated Business Planning (Ibp) (AMS Project)</span>,
 <span class="sc-ab270149-0 ePOHWr" color="#000" font-family="Inter" name="paragraph">Hết hạn trong 18 ngày</span>,
 <span class="sc-ab27

In [15]:
new_soup.find_all('span', attrs={'name': 'paragraph'})[0].text.strip()

'Hết hạn trong 18 ngày'

In [10]:
Location = new_soup.find_all('span', attrs={'name': 'paragraph'})[-1].text.strip()
Location

'Hồ Chí Minh, Hà Nội'

In [16]:
# function 4: nơi làm việc

def get_local(new_soup):
    try:
        local = new_soup.find_all('span', attrs={'name': 'paragraph'})[-1].text.strip()
        
    except AttributeError:
        local = None

    return local

In [17]:
local = get_local(new_soup)
local

'Hà Nội'

#### 3.2.1.5	Deadline date
tính hạn nộp CV (yyyy-mm-dd)

- T check theo ngày đăng gần nhất, và ngày đăng xa nhất
    - thì họ ghi các dạng
    - Hết hạn trong 21 giờ
    - Hết hạn trong 1 tháng
    - Hết hạn trong  2 ngày

In [20]:
# function 5: Tính toán ngày hết hạn nộp CV (Job Deadline)

def get_deadline_date(new_soup):
    try:
        # 1. Lấy đoạn text thô (ví dụ: "Hết hạn trong 21 giờ")
        # Lưu ý: Dùng .find sẽ an toàn hơn .find_all()[0] nếu trang web bị thiếu thẻ
        raw_text = new_soup.find('span', attrs={'name': 'paragraph'}).text.strip()
        
        # 2. Dùng Regex để tách lấy con số trong chuỗi
        match = re.search(r'\d+', raw_text)
        if not match:
            return None
        
        number = int(match.group())
        now = datetime.datetime.now() # Lấy thời điểm hiện tại lúc crawl
        
        # 3. Logic tính toán dựa trên đơn vị thời gian
        if 'giờ' in raw_text:
            # Nếu hết hạn trong vài giờ, nghĩa là hạn chót vẫn là ngày hôm nay
            deadline = now
        elif 'ngày' in raw_text:
            # Cộng thêm số ngày tương ứng
            deadline = now + datetime.timedelta(days=number)
        elif 'tháng' in raw_text:
            # Quy đổi tạm thời 1 tháng = 30 ngày
            deadline = now + datetime.timedelta(days=number * 30)
        else:
            # Các trường hợp khác không xác định
            return None
            
        # 4. Trả về định dạng ngày chuẩn YYYY-MM-DD để sau này dễ vẽ biểu đồ
        return deadline.strftime('%Y-%m-%d')

    except (AttributeError, IndexError, Exception):
        # Nếu không tìm thấy thẻ hoặc có lỗi tính toán, trả về None
        return None

In [21]:
job_deadline_date = get_deadline_date(new_soup)
job_deadline_date

'2026-04-23'

#### 3.2.1.6	Salary

In [12]:
salary_level = new_soup.find_all('span', attrs={'name': 'label'})[0].text.strip()
salary_level

'Thương lượng'

In [21]:
# function 6: lấy mức lương

def get_salary(new_soup):
    try:
        salary = new_soup.find_all('span', attrs={'name': 'label'})[0].text.strip()
        
    except AttributeError:
        salary = None

    return salary

In [22]:
salary = get_salary(new_soup)
salary

'Thương lượng'

### 3.2.2. THÔNG TIN VIỆC LÀM

#### 3.2.2.7	Posted Date

In [65]:
label = new_soup.find("label", string="NGÀY ĐĂNG")
label

<label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">NGÀY ĐĂNG</label>

In [ ]:
date_posted = label.find_next("p").get_text(strip=True)
date_posted

In [23]:
# function 7: lấy ngày mới nhất cập nhật thông tin job

def get_date_posted(new_soup):
    try:
        label = new_soup.find("label", string="NGÀY ĐĂNG")
        date_posted = label.find_next("p").get_text(strip=True)
        
    except AttributeError:
        date_posted = None

    return date_posted

In [24]:
date_posted = get_date_posted(new_soup)
date_posted

'24/03/2026'

#### 3.2.2.8	Category Job

In [ ]:
# lấy thông tin ngành nghề  
label_category = new_soup.find("label", string="NGÀNH NGHỀ")
category = label_category.find_next("p")
category

<p class="sc-ab270149-0 cLLblL" color="#333333" font-family="Inter" name="paragraph"><span class="sc-7bf5461f-3 bPjxRh">Công Nghệ Thông Tin/Viễn Thông</span> &gt; <span class="sc-7bf5461f-3 bPjxRh">Data Engineer/Data Analyst/AI</span></p>

In [25]:
# lấy thông tin ngành nghề  
label_category = new_soup.find("label", string="NGÀNH NGHỀ")
category = label_category.find_next("p").find_all(string=True)[-1].strip()
category

'Phân Tích Kinh Doanh/Phân Tích Hệ Thống'

In [26]:
# function 8: lấy thông tin ngành nghề  
# (Công Nghệ Thông Tin/Viễn Thông > Data Engineer/Data Analyst/AI)

def get_category(new_soup):
    try:
        label_category = new_soup.find("label", string="NGÀNH NGHỀ")
        category = label_category.find_next("p").find_all(string=True)[-1].strip()
        
    except AttributeError:
        category = None

    return category

In [27]:
category = get_category(new_soup)
category

'Phân Tích Kinh Doanh/Phân Tích Hệ Thống'

#### 3.2.2.9	Lĩnh vực

In [14]:
# lấy thông tin lĩnh vực công ty
label_field = new_soup.find("label", string="LĨNH VỰC")
field = label_field.find_next("p")
field

<p class="sc-ab270149-0 cLLblL" color="#333333" font-family="Inter" name="paragraph"><span class="sc-7bf5461f-3 bPjxRh">Tài Chính</span></p>

In [19]:
label_field = new_soup.find("label", string="LĨNH VỰC")
field = label_field.find_next("p").get_text(strip=True)
field

'Phần Mềm CNTT/Dịch vụ Phần mềm'

In [28]:
# function 9: lĩnh vực

def get_field(new_soup):
    try:
        label_field = new_soup.find("label", string="LĨNH VỰC")
        field = label_field.find_next("p").get_text(strip=True)
        
    except AttributeError:
        field = None

    return field

In [29]:
field = get_field(new_soup)
field

'Phần Mềm CNTT/Dịch vụ Phần mềm'

#### 3.2.2.10	Số năm kinh nghiệm

In [20]:
label_exprience = new_soup.find("label", string="SỐ NĂM KINH NGHIỆM TỐI THIỂU")
exprience = label_exprience.find_next("p").get_text(strip=True)
exprience

'1'

In [ ]:
# function 10: year - Exprience

def get_exprience(new_soup):
    try:
        label_exprience = new_soup.find("label", string="SỐ NĂM KINH NGHIỆM TỐI THIỂU")
        exprience = label_exprience.find_next("p").get_text(strip=True)
        
    except AttributeError:
        exprience = None

    return exprience

In [ ]:
exprience = get_exprience(new_soup)
exprience

'5'

#### 3.2.2.11	Trình độ học vấn

Đoạn này vướng nút Button "Xem thêm"

In [38]:
# Ko hề hiển thị phần Trình độ học phần, và các phần bên trong "XEM THÊM"

label_edu = new_soup.find_all("label", attrs={'name':"label"})
label_edu

[<label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">NGÀY ĐĂNG</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">CẤP BẬC</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">NGÀNH NGHỀ</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">KỸ NĂNG</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">LĨNH VỰC</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">NGÔN NGỮ TRÌNH BÀY HỒ SƠ</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">SỐ NĂM KINH NGHIỆM TỐI THIỂU</label>,
 <label class="sc-ab270149-0 dfyRSX" color="#939393" font-family="Inter" name="label">QUỐC TỊCH</label>]

In [42]:
new_soup.find_all("button", string="Xem thêm")

[<button aria-label="Xem thêm" class="sc-bd699a4b-0 eOtpMH btn-highlight btn-md clickable" type="button">Xem thêm</button>,
 <button aria-label="Xem thêm" class="sc-bd699a4b-0 eOtpMH btn-highlight btn-md clickable" style="width:100%" type="button">Xem thêm</button>]

In [44]:
new_soup.find("h2", string="Thông tin việc làm").find_next("button", string="Xem thêm")

<button aria-label="Xem thêm" class="sc-bd699a4b-0 eOtpMH btn-highlight btn-md clickable" style="width:100%" type="button">Xem thêm</button>

In [37]:
label_edu = new_soup.find("label", string="TRÌNH ĐỘ HỌC VẤN TỐI THIỂU")
label_edu

In [40]:
new_soup.find("label", string="TRÌNH ĐỘ HỌC VẤN TỐI THIỂU")

In [ ]:
label_edu = new_soup.find("label", string="TRÌNH ĐỘ HỌC VẤN TỐI THIỂU")
edu = label_edu.find_next("p").get_text(strip=True)
edu

In [45]:
# function 11: Educatiom

def get_edu(new_soup):
    try:
        label_edu = new_soup.find("label", string="TRÌNH ĐỘ HỌC VẤN TỐI THIỂU")
        edu = label_edu.find_next("p").get_text(strip=True)

    except AttributeError:
        edu = None

    return edu

In [46]:
edu = get_edu(new_soup)
edu

'Không giới hạn'

#### 3.2.2.12	Số lượng tuyển dụng

In [ ]:
label_hire_num = new_soup.find("label", string="SỐ LƯỢNG TUYỂN DỤNG")
hire_num = label_hire_num.find_next("p").get_text(strip=True)
hire_num

'Không hiển thị'

In [51]:
# function 12: hiring numbers

def get_hire_num(new_soup):
    try:
        label_hire_num = new_soup.find("label", string="SỐ LƯỢNG TUYỂN DỤNG")
        hire_num = label_hire_num.find_next("p").get_text(strip=True)

    except AttributeError:
        hire_num = None

    return hire_num

In [52]:
hire_num = get_hire_num(new_soup)
hire_num

'10'

#### 3.2.2.13	Loại công việc ( full / partime ) => Job type

In [ ]:
label_job_type = new_soup.find("label", string="LOẠI HÌNH LÀM VIỆC")
job_type = label_job_type.find_next("p").get_text(strip=True)
job_type

'Toàn thời gian'

In [53]:
# function 13: Job type

def get_job_type(new_soup):
    try:
        label_job_type = new_soup.find("label", string="LOẠI HÌNH LÀM VIỆC")
        job_type = label_job_type.find_next("p").get_text(strip=True)

    except AttributeError:
        job_type = None

    return job_type

In [54]:
job_type = get_job_type(new_soup)
job_type

'Toàn thời gian'

#### 3.2.2.14	Cấp bậc (nhân viên / intern)

In [35]:
label_job_level = new_soup.find("label", string="CẤP BẬC")
job_level = label_job_level.find_next("p").get_text(strip=True)
job_level


'Nhân viên'

In [55]:
# function 14: Job Level

def get_job_level(new_soup):
    try:
        label_job_level = new_soup.find("label", string="CẤP BẬC")
        job_level = label_job_level.find_next("p").get_text(strip=True)

    except AttributeError:
        job_level = None

    return job_level

In [56]:
job_level = get_job_level(new_soup)
job_level

'Nhân viên'

#### 3.2.2.15	Kỹ năng Skill -> text mining

In [39]:
label_skill = new_soup.find("label", string="KỸ NĂNG")
skill = label_skill.find_next("p").get_text(strip=True)
skill


'Kiểm Thử Phần Mềm, Công Nghệ Thông Tin, Xử Lý Công Việc, Kỹ năng làm việc nhóm, Tư duy logic'

In [58]:
# function 15: Skill, sau dùng Chart đám mây ngôn ngữ

def get_skill(new_soup):
    try:
        label_skill = new_soup.find("label", string="KỸ NĂNG")
        skill = label_skill.find_next("p").get_text(strip=True)

    except AttributeError:
        skill = None

    return skill

In [59]:
skill = get_skill(new_soup)
skill

'Agile Scrum, AI/ML, Communication Skills, Database Design, Requirements Analysis'

#### 3.2.2.16	Độ tuổi mong muốn
- "15-60", "20-35", "18-25 tuổi"
- match = re.search(r'(\d+)[-\s]*(\d+)', text)

1. **Age distribution**: Data jobs cần bao nhiêu tuổi?
2. **Salary vs Age**: Fresher 18-25 lương thấp → Senior 35+ lương cao
3. **Role vs Age**: Entry vs Engineer vs Manager
4. **Market trend**: Data market trẻ hóa hay senior hóa?

In [ ]:
label_age = new_soup.find("label", string="ĐỘ TUỔI MONG MUỐN")
age = label_age.find_next("p").get_text(strip=True)
age

'15-60'

In [61]:
# function 16: Độ tuổi min - max 

def get_age(new_soup):
    try:
        label_age = new_soup.find("label", string="ĐỘ TUỔI MONG MUỐN")
        age = label_age.find_next("p").get_text(strip=True)

    except AttributeError:
        age = None

    return age

In [ ]:
age = get_age(new_soup)
age

### 3.2.3.      MÔ TẢ CÔNG VIỆC

In [14]:
# 1. Tìm thẻ h2 có chứa chữ "Mô tả công việc"
# Dùng lambda để tìm kiếm linh hoạt hơn (phòng trường hợp có khoảng trắng thừa)
h2_anchor = new_soup.find('h2', string=lambda t: t and "Mô tả công việc" in t)

if h2_anchor:
    # 2. Tìm thẻ div chứa nội dung nằm ngay sau thẻ h2 đó
    # find_next('div') sẽ tìm thẻ div đầu tiên xuất hiện sau h2_anchor
    content_div = h2_anchor.find_next(lambda tag: tag.name == 'div' and tag.find('p'))
    
    if content_div:
        # 3. Lấy text, dùng dấu cách làm separator để tránh dính chữ
        raw_text = content_div.get_text(separator=' ', strip=True)
        
        # 4. Làm sạch cho Text Mining:
        # - Loại bỏ các ký tự đặc biệt thường gặp trong list (•, -, *)
        # - Chuẩn hóa khoảng trắng (biến nhiều dấu cách/xuống dòng thành 1 dấu cách)
        clean_text = re.sub(r'[•\*]', '', raw_text) # Xóa dấu chấm tròn, gạch ngang, dấu sao
        final_text = ' '.join(clean_text.split())
        
        print(final_text)

Mô tả công việc Thu thập yêu cầu nghiệp vụ và yêu cầu hệ thống từ khách hàng. Hỗ trợ khách hàng làm rõ nhu cầu, đề xuất ý tưởng/giải pháp phù hợp. Soạn thảo tài liệu URD/SRS; viết đặc tả Use Case chi tiết, User Stories và wireframe cho các use case phức tạp (khi cần). Chuyển giao kiến thức và yêu cầu cho đội phát triển. Hỗ trợ PM về phạm vi, ước lượng và lập kế hoạch dự án/sản phẩm. Hỗ trợ PM trong quy trình ký duyệt yêu cầu và xử lý change request. Đóng vai trò cầu nối làm rõ yêu cầu giữa khách hàng và team nội bộ; tham gia review và thúc đẩy giao tiếp hiệu quả. Hỗ trợ UAT: kiểm thử, xác minh bug/issue với khách hàng. Thực hiện các nhiệm vụ khác theo phân công của Line Manager / Product Manager


In [16]:
# function 17: Mô tả công việc

def get_job_description(new_soup):
    try:
        # 1. Tìm thẻ h2 làm mỏ neo
        h2_anchor = new_soup.find('h2', string=lambda t: t and "Mô tả công việc" in t)
        
        # 2. Tìm thẻ div tiếp theo có chứa thẻ <p>
        content_div = h2_anchor.find_next(lambda tag: tag.name == 'div' and tag.find('p'))
        
        # 3. Lấy text và làm sạch
        raw_text = content_div.get_text(separator=' ', strip=True)
        clean_text = re.sub(r'[•\*]', '', raw_text)
        job_description = ' '.join(clean_text.split())

    except AttributeError:
        job_description = None

    return job_description

In [17]:
job_description = get_job_description(new_soup)
job_description

'Mô tả công việc Thu thập yêu cầu nghiệp vụ và yêu cầu hệ thống từ khách hàng. Hỗ trợ khách hàng làm rõ nhu cầu, đề xuất ý tưởng/giải pháp phù hợp. Soạn thảo tài liệu URD/SRS; viết đặc tả Use Case chi tiết, User Stories và wireframe cho các use case phức tạp (khi cần). Chuyển giao kiến thức và yêu cầu cho đội phát triển. Hỗ trợ PM về phạm vi, ước lượng và lập kế hoạch dự án/sản phẩm. Hỗ trợ PM trong quy trình ký duyệt yêu cầu và xử lý change request. Đóng vai trò cầu nối làm rõ yêu cầu giữa khách hàng và team nội bộ; tham gia review và thúc đẩy giao tiếp hiệu quả. Hỗ trợ UAT: kiểm thử, xác minh bug/issue với khách hàng. Thực hiện các nhiệm vụ khác theo phân công của Line Manager / Product Manager'

### 3.2.4.      YÊU CẦU

In [15]:

# 1. Tìm thẻ h2 làm mỏ neo
h2_anchor = new_soup.find('h2', string=lambda t: t and "Yêu cầu công việc" in t)

if h2_anchor:
    # 2. Tìm thẻ div tiếp theo mà có chứa các thẻ <p> bên trong
    # Đây là cách lọc bỏ banner hiệu quả vì banner thường không dùng thẻ <p> cho nội dung chính
    content_div = h2_anchor.find_next(lambda tag: tag.name == 'div' and tag.find('p'))

    if content_div:
        # 3. Lấy text với separator là dấu cách để không dính chữ
        raw_text = content_div.get_text(separator=' ', strip=True)
        
        # 4. Làm sạch cho Text Mining
        # Loại bỏ các ký tự bullet points (•, -, *)
        clean_text = re.sub(r'[•\*]', '', raw_text)
        
        # Chuẩn hóa khoảng trắng (biến xuống dòng, tab, nhiều dấu cách thành 1 dấu cách duy nhất)
        final_text = ' '.join(clean_text.split())
        
        print(final_text)

Chuyên môn trong lĩnh vực Có nền tảng kiến thức CNTT (IT knowledge base). Tối thiểu 5 năm kinh nghiệm phân tích yêu cầu (requirements analysis). Tối thiểu 5 năm kinh nghiệm phân tích yêu cầu (requirements analysis). Ưu tiên có hiểu biết domain: AI/ML, Computer Vision, v.v. Kinh nghiệm Kiến thức cơ bản về CSDL (database schema) và viết truy vấn SQL đơn giản. Hiểu biết về phương pháp và kỹ thuật kiểm thử (testing approaches & techniques) Kinh nghiệm làm việc trong môi trường Agile/Scrum. Tham gia daily standup, sprint planning và sprint review để cập nhật tiến độ sản phẩm / dự án. Kỹ năng cá nhân Kỹ năng giao tiếp và phối hợp (communication & interpersonal) tốt. Khả năng viết tài liệu URD/SRS rõ ràng, logic. Tư duy hệ thống, chủ động phát hiện rủi ro và đề xuất giải pháp. Mindset Chủ động, tinh thần “can-do”. Customer_Centric: Đặt người dùng/khách hàng làm trung tâm khi phân tích và ra quyết định Tư duy tổng thể: hiểu bức tranh end-to-end, bối cảnh tổ chức, ràng buộc và tác động đến các 

In [18]:
# function 18: Yêu cầu công việc

def get_job_requirements(new_soup):
    try:
        # 1. Tìm thẻ h2 làm mỏ neo
        h2_anchor = new_soup.find('h2', string=lambda t: t and "Yêu cầu công việc" in t)
        
        # 2. Tìm thẻ div tiếp theo có chứa thẻ <p>
        content_div = h2_anchor.find_next(lambda tag: tag.name == 'div' and tag.find('p'))
        
        # 3. Lấy text và làm sạch
        raw_text = content_div.get_text(separator=' ', strip=True)
        clean_text = re.sub(r'[•\*]', '', raw_text)
        job_requirements = ' '.join(clean_text.split())

    except AttributeError:
        job_requirements = None

    return job_requirements

In [19]:
job_requirements = get_job_requirements(new_soup)
job_requirements

'Chuyên môn trong lĩnh vực Có nền tảng kiến thức CNTT (IT knowledge base). Tối thiểu 5 năm kinh nghiệm phân tích yêu cầu (requirements analysis). Tối thiểu 5 năm kinh nghiệm phân tích yêu cầu (requirements analysis). Ưu tiên có hiểu biết domain: AI/ML, Computer Vision, v.v. Kinh nghiệm Kiến thức cơ bản về CSDL (database schema) và viết truy vấn SQL đơn giản. Hiểu biết về phương pháp và kỹ thuật kiểm thử (testing approaches & techniques) Kinh nghiệm làm việc trong môi trường Agile/Scrum. Tham gia daily standup, sprint planning và sprint review để cập nhật tiến độ sản phẩm / dự án. Kỹ năng cá nhân Kỹ năng giao tiếp và phối hợp (communication & interpersonal) tốt. Khả năng viết tài liệu URD/SRS rõ ràng, logic. Tư duy hệ thống, chủ động phát hiện rủi ro và đề xuất giải pháp. Mindset Chủ động, tinh thần “can-do”. Customer_Centric: Đặt người dùng/khách hàng làm trung tâm khi phân tích và ra quyết định Tư duy tổng thể: hiểu bức tranh end-to-end, bối cảnh tổ chức, ràng buộc và tác động đến các

### 3.2.5.	PHÚC LỢI

In [22]:
# function 20: Phúc lợi (Chỉ lấy Raw Text có cấu trúc)

def get_benefits(new_soup):
    try:
        h2_anchor = new_soup.find('h2', string=lambda t: t and "phúc lợi" in t.lower())
        container = h2_anchor.find_parent('div') 
        benefit_blocks = container.find_all('div', attrs={'data-benefit-name': True})
        
        benefit_list = []
        for block in benefit_blocks:
            title = block.find('p').get_text(strip=True)
            # Chúng ta chỉ lấy cái nhãn (Category) thôi cho nhẹ dữ liệu
            benefit_list.append(title)
        
        return "|".join(benefit_list) if benefit_list else None
    except:
        return None

In [23]:
benefits = get_benefits(new_soup)
benefits

'Thưởng|Chăm sóc sức khoẻ'

In [1]:
import time
import re
from bs4 import BeautifulSoup
from selenium import webdriver

# 1. Khởi tạo lại trình duyệt (nếu driver trước đó đã bị đóng)
driver = webdriver.Chrome()

# 2. Vào thẳng link job bị lỗi
job_link = 'https://www.vietnamworks.com/business-analyst-intern--2025657-jv?source=searchResults&searchType=2&placement=2025707&sortBy=latest&qs=0'
print(f"Bắt đầu load link: {job_link}")
driver.get(job_link)
time.sleep(3) # Đợi trang web render DOM xong

# 3. Lấy BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 4. Hàm cào Mô tả công việc - ĐÃ SỬA LỖI EXCEL
def get_job_description_fixed(soup):
    try:
        h2_anchor = soup.find('h2', string=lambda t: t and "Mô tả công việc" in t)
        content_div = h2_anchor.find_next(lambda tag: tag.name == 'div' and tag.find('p'))
        raw_text = content_div.get_text(separator=' ', strip=True)
        
        # Regex cũ của bạn: loại bỏ dấu chấm đen và sao
        clean_text = re.sub(r'[•\*]', '', raw_text)
        final_text = ' '.join(clean_text.split())
        
        # --- CÁCH KHẮC PHỤC LỖI #NAME? CHO EXCEL ---
        # Kiểm tra xem chuỗi có bắt đầu bằng các ký tự nguy hiểm cho Excel không: -, =, +, @
        if final_text.startswith(('-', '=', '+', '@')):
            # Thêm một dấu nháy đơn ' ở đầu câu. 
            # Excel khi thấy dấu ' ở đầu nó sẽ tự hiểu đây là đoạn text thuần túy chứ ko phải công thức. 
            # Dấu ' này sẽ bị tàng hình khi bạn mở bằng Excel, rất đẹp mắt!
            final_text = "'" + final_text
            
            # (Hoặc bạn có thể dùng cách thêm 1 khoảng trắng ở đầu: final_text = " " + final_text)
            
        return final_text
    except AttributeError as e:
        return f"Lỗi không tìm thấy thẻ: {e}"

# 5. In thử kết quả để đối chiếu
ket_qua = get_job_description_fixed(soup)
print("\n=== KẾT QUẢ MÔ TẢ CÔNG VIỆC CÀO ĐƯỢC ===")
print(ket_qua)

driver.quit()


Bắt đầu load link: https://www.vietnamworks.com/business-analyst-intern--2025657-jv?source=searchResults&searchType=2&placement=2025707&sortBy=latest&qs=0

=== KẾT QUẢ MÔ TẢ CÔNG VIỆC CÀO ĐƯỢC ===
'- Nhập liệu và cập nhật dữ liệu cho các ngành may trên hệ thống ERP. - Theo dõi các đơn hàng chưa được đóng trên hệ thống và thực hiện các bước xử lý sau khi đơn hàng đã xuất hàng. - Kiểm tra và đảm bảo tính chính xác của dữ liệu được nhập vào hệ thống. - Hỗ trợ các bộ phận liên quan trong việc cập nhật, kiểm tra và quản lý dữ liệu trên hệ thống ERP. - Thực hiện các công việc khác theo sự phân công của quản lý.
